# GCP Pose Estimation — Kaggle Training

## Before running: download the data as a ZIP and upload to Kaggle

**1. Download the Drive folder as ZIP (on your laptop)**
- Open [drive.google.com](https://drive.google.com) → find the shared GCP data folder
- Right-click the folder → **Download**
- Chrome will zip it and download. If the folder is >2 GB, Drive splits it into multiple ZIPs — download **all** of them.

**2. Upload the ZIP(s) as a Kaggle dataset**
- kaggle.com → your profile → **Datasets** → **New Dataset**
- Upload all ZIP file(s) → title: `skylark-gcp-data` → visibility: **Private** → Create

**3. Add that dataset as input to this notebook**
- In this notebook → **Add Input** (top right) → Your Datasets → `skylark-gcp-data` → Add

Then run cells top to bottom. GPU T4×2 + Internet must both be ON.

In [ ]:
# 1. Clone repo + install dependencies
!git clone https://github.com/J4KE-B/skylark-gcp gcp
%cd gcp
!pip -q install -r requirements.txt

In [ ]:
# 2. Extract data from uploaded ZIP(s)
import os, shutil, yaml, glob, json, zipfile

zip_files = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True))
assert zip_files, (
    "No ZIP found in /kaggle/input/\n"
    "Fix: Add Input -> Your Datasets -> skylark-gcp-data"
)
print(f"Found {len(zip_files)} ZIP(s): {[os.path.basename(z) for z in zip_files]}")

os.makedirs('data', exist_ok=True)
for zp in zip_files:
    print(f"Extracting {os.path.basename(zp)} ...")
    with zipfile.ZipFile(zp, 'r') as zf:
        zf.extractall('data/')
print("Extraction complete.")
print("data/ contents:", sorted(os.listdir('data')))

# Rename train/ -> train_dataset/ and test/ -> test_dataset/ if Drive stripped the suffix
for short, full in [('train', 'train_dataset'), ('test', 'test_dataset')]:
    if os.path.isdir(f'data/{short}') and not os.path.isdir(f'data/{full}'):
        os.rename(f'data/{short}', f'data/{full}')
        print(f"Renamed data/{short}/ -> data/{full}/")

# Flatten any remaining Drive wrapper folders
candidates = [d for d in os.listdir('data')
              if os.path.isdir(f'data/{d}') and d not in ('train_dataset', 'test_dataset')]
for name in candidates:
    src = f'data/{name}'
    print(f"Flattening {src}/ -> data/")
    for item in os.listdir(src):
        dst = f'data/{item}'
        if not os.path.exists(dst):
            shutil.move(f'{src}/{item}', dst)
    if not os.listdir(src):
        os.rmdir(src)

print("data/ contents after normalise:", sorted(os.listdir('data')))
assert os.path.exists('data/gcp_marks.json'), 'FAIL: gcp_marks.json missing'
assert os.path.isdir('data/train_dataset'),   'FAIL: train_dataset/ missing'
assert os.path.isdir('data/test_dataset'),    'FAIL: test_dataset/ missing'

# Patch gcp_marks.json keys to match sanitized filenames ('&' -> 'and')
raw = json.load(open('data/gcp_marks.json'))
patched = {k.replace('&', 'and'): v for k, v in raw.items()}
if patched != raw:
    json.dump(patched, open('data/gcp_marks.json', 'w'))
    print(f"Patched {sum(1 for k in raw if '&' in k)} label keys ('&' -> 'and')")

# Wire config paths
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['paths'] = {'label_file': 'data/gcp_marks.json',
                'train_dir':  'data/train_dataset',
                'test_dir':   'data/test_dataset',
                'output_dir': 'outputs'}
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'))

# Verify coverage
labels = json.load(open('data/gcp_marks.json'))
found  = sum(1 for k in labels if os.path.exists(f"data/train_dataset/{k}"))
pct    = 100 * found // len(labels)
print(f"Train images on disk: {found}/{len(labels)} ({pct}%)")
print("OK — enough to train." if found >= len(labels) * 0.8 else
      "WARNING: <80% images — check that all ZIPs were uploaded and re-run this cell.")


In [ ]:
# 3. SMOKE: 3 epochs, then inspect predictions schema before committing GPU time
import yaml
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 3
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import json; p = json.load(open('predictions.json'))
k = next(iter(p)); print(k, '->', p[k]); print('total:', len(p))

In [ ]:
# 4. FULL run: restore 40 epochs
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 40
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml

In [ ]:
# 5. Final predictions + download artifacts
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import pandas as pd; pd.read_csv('outputs/training_log.csv').tail()

## Artifacts to download

From the Kaggle Output panel (right sidebar), download these three files:

- `gcp/outputs/best.pt` — model weights (upload to Drive, copy the shareable link)
- `gcp/outputs/training_log.csv` — open it, read the last row for PCK@10/25/50 + Macro-F1
- `gcp/predictions.json` — this is your submission file for Skylark

Then update `README.md` in the repo:
1. Fill in the Results table with the PCK/F1 numbers
2. Add the Drive link for `best.pt`
3. `git add README.md && git commit -m "docs: add training results and weights link" && git push`